# 2. Model development, frozen selection, and inference

This notebook owns all model fitting and prediction generation. It first completes **validation-only portfolio selection** using the shared simulator, then freezes the entry before assignment evaluation or competition fitting. This ordering permits all three notebooks to run sequentially while preserving independence.

The assignment refit uses complete outcomes before January 2024. The separate competition refit uses complete outcomes before January 2026. No 2026 row enters fitting, tuning, risk thresholds, feature importance, or strategy selection.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/terminal34-datascience-bootcamp/stock-market-ml/blob/main/02_training_inference.ipynb)

## Run locally or in Google Colab

In Colab, select a **Python 3 CPU runtime**, then run the setup cell below and authorize its Google Drive mount. It loads the shared code from GitHub and installs the runtime dependencies. Use the same `EXPERIMENT_NAME` and `COLAB_SMOKE_TEST` values in notebooks **1 → 2 → 3**; each can run in a separate session. Your datasets, models, predictions, and frozen selection are saved directly under `My Drive/stock-market-ml/<experiment>/real/` (or `smoke/`), so they survive runtime resets. Save a notebook copy to Drive if you want to keep your notes and cell outputs too.

**Private GitHub repository:** add a `GITHUB_TOKEN` in Colab's Secrets panel (key icon), using a fine-grained token with **Contents: Read-only** access to this repository, and enable notebook access in each notebook. Organization approval may be required. Do not paste the token into a code cell. Setup passes it only to Git subprocesses and does not save it in artifacts or Git remote URLs. If Colab cannot open the private GitHub link, download the `.ipynb` from GitHub and upload it to Colab; the same secret enables the code download.

The first notebook records a Git commit and exact package versions; later notebooks reuse them. If setup asks for a runtime restart after installing packages, restart the session and rerun setup. Run one notebook at a time for a given experiment. A missing-handoff error means you need to finish the preceding notebook with the same settings. On a local machine, setup preserves your environment and artifact path; install `requirements.txt` first.

Changing the experiment folder after seeing 2026 results does not restore holdout independence. To try this Colab port without revisiting the real holdout, set `COLAB_SMOKE_TEST=True` in all three notebooks. Your existing local artifacts are not uploaded automatically.

In [ ]:
# Use the SAME values in all three Colab notebooks.
EXPERIMENT_NAME = 'competition-v1'
COLAB_SMOKE_TEST = False  # True = synthetic data and small models, stored separately.

"""Standard-library-only bootstrap, embedded into each notebook before third-party imports."""
from pathlib import Path
import base64
import importlib.metadata
import importlib.util
import json
import os
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/terminal34-datascience-bootcamp/stock-market-ml.git'
RUNTIME_PACKAGES = {
    'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
    'scikit-learn': 'sklearn', 'xgboost': 'xgboost', 'yfinance': 'yfinance',
    'pyarrow': 'pyarrow', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'joblib': 'joblib',
}


def in_colab():
    try:
        return importlib.util.find_spec('google.colab') is not None
    except (ImportError, ValueError):
        return False


def require_handoff(artifacts, notebook_number):
    required = {
        1: [],
        2: ['data/metadata.json'],
        3: ['frozen_experiment.json', 'competition_evaluation.json',
            'predictions/assignment.parquet', 'predictions/competition.parquet'],
    }
    if notebook_number not in required:
        raise ValueError('Notebook number must be 1, 2, or 3')
    missing = [name for name in required[notebook_number] if not (artifacts / name).is_file()]
    if missing:
        raise RuntimeError(
            f'Missing handoff files in {artifacts}: {missing}. '
            f'Run Notebook {notebook_number - 1} first, with the same EXPERIMENT_NAME and smoke setting.'
        )


def github_git_environment(token=None):
    # Headers live only in child-process environments, never URLs, arguments, or .git/config.
    environment = {key: value for key, value in os.environ.items()
                   if not key.startswith('GIT_TRACE') and key != 'GIT_CURL_VERBOSE'}
    environment['GIT_TERMINAL_PROMPT'] = '0'
    if token:
        encoded = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        environment.update({'GIT_CONFIG_COUNT': '1',
                            'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
                            'GIT_CONFIG_VALUE_0': f'Authorization: Basic {encoded}'})
    return environment


def prepare_colab_checkout(artifacts, project, repository_url=REPOSITORY_URL, token=None):
    """Use one source revision and resolved package set for all three sessions."""
    artifacts, project = Path(artifacts), Path(project)
    artifacts.mkdir(parents=True, exist_ok=True)
    manifest_path = artifacts / 'colab_environment.json'
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else None
    python_version = f'{sys.version_info.major}.{sys.version_info.minor}'
    if manifest and (manifest['repository_url'] != repository_url or manifest['python'] != python_version):
        raise RuntimeError('This experiment belongs to a different repository or Python version. '
                           'Use the recorded runtime version to reuse its models.')
    git_environment = github_git_environment(token)
    if not project.exists():
        try:
            subprocess.check_call(['git', 'clone', '--quiet', repository_url, str(project)], env=git_environment)
        except subprocess.CalledProcessError as error:
            raise RuntimeError('Cannot clone the repository. For this private repo, add a GITHUB_TOKEN '
                               'Colab secret with Contents: Read access, enable notebook access, and rerun setup.') from error
    if not (project / '.git').is_dir():
        raise RuntimeError(f'{project} exists but is not a Git checkout; choose a fresh Colab session.')
    git = ['git', '-C', str(project)]
    origin = subprocess.check_output(git + ['remote', 'get-url', 'origin'], text=True).strip()
    if origin != repository_url:
        raise RuntimeError(f'Refusing to use an unrelated checkout at {project}')
    if subprocess.check_output(git + ['status', '--porcelain'], text=True).strip():
        raise RuntimeError('The Colab checkout has local edits. Save them before using a fresh runtime; '
                           'setup will not overwrite your work.')
    ref = manifest['commit'] if manifest else 'main'
    subprocess.check_call(git + ['fetch', '--quiet', 'origin', ref], env=git_environment)
    commit = subprocess.check_output(git + ['rev-parse', 'FETCH_HEAD'], text=True).strip()
    subprocess.check_call(git + ['checkout', '--quiet', '--detach', commit])
    requirements = project / 'requirements-colab.txt'
    if not requirements.is_file():
        raise RuntimeError('This GitHub revision does not include Colab support. Publish the updated project first.')
    if manifest:
        requirements = artifacts / 'colab-runtime-requirements.txt'
        requirements.write_text(''.join(f'{name}=={version}\n' for name, version in manifest['packages'].items()))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(requirements)])
    resolved = {name: importlib.metadata.version(name) for name in RUNTIME_PACKAGES}
    if manifest and resolved != manifest['packages']:
        raise RuntimeError('Installed packages do not match the saved Colab environment')
    if manifest is None:
        manifest = {'repository_url': repository_url, 'commit': commit,
                    'python': python_version, 'packages': resolved}
        temporary = manifest_path.with_suffix('.json.tmp')
        temporary.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n')
        temporary.replace(manifest_path)
        (artifacts / 'colab-runtime-requirements.txt').write_text(
            ''.join(f'{name}=={version}\n' for name, version in resolved.items()))
    changed = [name for name, module in RUNTIME_PACKAGES.items()
               if module in sys.modules and getattr(sys.modules[module], '__version__', resolved[name]) != resolved[name]]
    if changed:
        raise RuntimeError(f'Installed versions changed for already-imported packages: {changed}. '
                           'Choose Runtime > Restart session, then run this setup cell again.')
    print('Pinned GitHub revision:', commit)
    return project


def setup_notebook(notebook_number, experiment_name='competition-v1', colab_smoke=False):
    if in_colab():
        # Authorization is performed by Colab's own Drive prompt, in the user's runtime.
        from google.colab import drive, userdata
        if not experiment_name or experiment_name in ['.', '..'] or any(c in experiment_name for c in '/\\'):
            raise ValueError('EXPERIMENT_NAME must be one folder name')
        drive.mount('/content/drive')
        artifacts = Path('/content/drive/MyDrive/stock-market-ml') / experiment_name / ('smoke' if colab_smoke else 'real')
        require_handoff(artifacts, notebook_number)
        try:
            token = userdata.get('GITHUB_TOKEN')
        except userdata.SecretNotFoundError:
            token = None  # Public forks need no token; private-repo errors explain the setup.
        except userdata.NotebookAccessError as error:
            raise RuntimeError('Enable notebook access for GITHUB_TOKEN in the Colab Secrets panel.') from error
        project = prepare_colab_checkout(artifacts, Path('/content/stock-market-ml'), token=token)
        os.environ['STOCK_ML_ARTIFACTS'] = str(artifacts)
        os.environ['STOCK_ML_SMOKE'] = '1' if colab_smoke else '0'
        os.chdir(project)
    else:
        project = Path.cwd()
        if not (project / 'stockml').is_dir():
            raise RuntimeError('For local execution, open the notebook from the repository root.')
        artifacts = Path(os.environ.get('STOCK_ML_ARTIFACTS', 'artifacts')).resolve()
        require_handoff(artifacts, notebook_number)
    sys.path.insert(0, str(project))
    os.environ.setdefault('MPLCONFIGDIR', str(project / '.cache' / 'matplotlib'))
    Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
    print('Artifact handoff folder:', artifacts)
    return project

PROJECT = setup_notebook(2, EXPERIMENT_NAME, COLAB_SMOKE_TEST)

import pandas as pd
from IPython.display import display, Markdown
from stockml.common import root, smoke, load_json

pd.set_option('display.max_columns', 20)
print('Artifacts:', root())
print('SYNTHETIC SMOKE TEST — NOT FINANCIAL RESULTS' if smoke() else 'REAL-DATA EXPERIMENT')

## Development protocol and bounded search

Split all stocks by calendar date, not stacked row number. Drop any training row whose target reaches the next period. Validation scoring excludes targets realized in 2024. Return models minimize validation RMSE; risk models maximize ROC-AUC.

- Random Forest: 300 trees; `max_depth` in `[4, 8, None]`, `min_samples_leaf` in `[10, 30]`.
- XGBoost: 300 trees; `max_depth` in `[2, 3, 5]`, `learning_rate` in `[0.03, 0.1]`.
- Both tasks use seed 42. Classification uses probability cutoff 0.5.
- Pooled models add a fixed one-hot ticker encoding. Individual models reuse each algorithm/task's selected pooled settings to bound compute; this is not a fully tuned individual-model comparison.
- High risk means future volatility strictly exceeds that stock's training 75th percentile. Each refit recomputes thresholds; compare models within the same experiment.

Baselines predict zero return and the training majority risk class. Zero-return directional accuracy counts a prediction correct only when the realized return is exactly zero; its rank correlation is undefined (N/A). Majority baseline AUC uses a constant training prevalence probability. Undefined single-class AUC is N/A; undefined precision/recall/F1 are reported as zero.

In [ ]:
from stockml.models import develop_and_freeze
frozen = develop_and_freeze()
display(pd.read_csv(root() / 'reports' / 'tuning.csv'))
validation_metrics = pd.read_csv(root() / 'reports' / 'validation_metrics.csv')
display(validation_metrics[validation_metrics.ticker == 'ALL'])
display(pd.read_csv(root() / 'reports' / 'validation_portfolios.csv'))
display(Markdown(f"Frozen entry: **{frozen['selected_entry']['strategy']}**. Freeze ID: `{frozen['freeze_id']}`."))

## Interpretation using validation data only

The four pooled charts show the ten largest permutation importances. Positive importance means shuffling the feature harmed validation performance; negative values can reflect noise or overfitting. Correlated indicators can share importance, so importance is not a causal claim.

The stock-specific table reports each model's three leading training impurity/gain importances. These are useful for comparing feature identities across stocks but are not numerically comparable to permutation scores. The ablation compares momentum plus ticker encoding with all features using identical rows and settings. More features help only if the validation result improves.

In [ ]:
from stockml.reporting import plot_importance
importance = plot_importance()
ablation = pd.read_csv(root() / 'reports' / 'feature_ablation.csv')
display(ablation)
from stockml.reporting import feature_discussion
feature_discussion()
display(validation_metrics[validation_metrics.ticker != 'ALL'])

## Assignment experiment: 2024–2025

Settings and portfolio rules are now fixed. Refit through 2023 using only targets completed before 2024. Predictions include the preceding close so the portfolio can trade at the first 2024 session's open. Predictive scoring itself uses only signal dates within the test period and complete targets ending within it.

Model artifacts include feature order, ticker encoding, thresholds, cutoffs, parameters, library versions, and dataset fingerprints. Every save is checked by reloading the trusted local artifact and comparing predictions exactly.

In [ ]:
from stockml.models import fit_final
assignment_bundle = fit_final('assignment')
assignment_metrics = pd.read_csv(root() / 'reports' / 'assignment_metrics.csv')
display(assignment_metrics[assignment_metrics.ticker == 'ALL'])
display(assignment_metrics[assignment_metrics.ticker != 'ALL'])

## Competition refit: through 2025, with no 2026 data access

Refit the already chosen configurations on complete outcomes through 2025. This creates a separate model bundle and fresh training-only volatility thresholds. It does not select a new strategy based on the assignment results.

In [ ]:
competition_bundle = fit_final('competition')
display(pd.Series({key: competition_bundle[key] for key in ['bundle_id', 'last_training_signal', 'last_training_target', 'freeze_id']}))

## Final holdout inference: 2026

This is the first model-stage access to 2026 features. Predictions are saved before realized outcomes are loaded for scoring. Recent rows without five subsequent sessions still receive predictions but do not enter predictive metrics.

Running this cell exposes the holdout results. Later changes to features, models, or strategy selection would make 2026 development data rather than a fully independent test. A repeated frozen evaluation is not a new independent experiment.

In [ ]:
from stockml.models import competition_inference
competition_predictions, competition_metrics = competition_inference(competition_bundle)
display(competition_metrics[competition_metrics.ticker == 'ALL'])
display(competition_metrics[competition_metrics.ticker != 'ALL'])
display(pd.Series(load_json(root() / 'competition_evaluation.json')))

## Handoff

`models/` holds the trusted local model bundles and readable metadata. `predictions/` holds separate validation, assignment, and competition predictions. `frozen_experiment.json` identifies the entry and rules selected on validation. Predictive metrics, tuning, importance, and ablation tables are under `reports/`.

No inference code in Notebook 3 refits these models. Continue there for portfolio evaluation and evidence-based conclusions.